### Introducing LangGraph!

#### Linear Pipeline

The simplest graph: a straight pipeline. Each node transforms the state and passes it to the next.

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env

api_key = os.getenv("OPENAI_API_KEY")

In [23]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

## 1. Define State — the shared data bag

In [4]:
class DocState(TypedDict):
    raw_text: str
    clean: str
    translated: str
    summary: str

## 2. Define Nodes — pure functions

In [7]:
def clean_text(s: DocState) -> dict:
    return {"clean": s["raw_text"].strip()}

def translate(s: DocState) -> dict:
    result = llm.invoke(f"Translate: {s['clean']}")
    return {"translated": result.content}

def summaries(s: DocState) -> dict:
    result = llm.invoke(f"Summarise: {s['translated']}")
    return {"summary": result.content}

## 3. Build Graph

In [8]:
builder = StateGraph(DocState)
builder.add_node("clean_text", clean_text)
builder.add_node("translate", translate)
builder.add_node("summaries", summaries)

## 4. Wire Edges

In [9]:
builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "translate")
builder.add_edge("translate", "summaries")
builder.add_edge("summaries", END)

In [13]:
graph = builder.compile()

In [19]:
from IPython.display import Markdown, display

display(Markdown(graph.get_graph().draw_mermaid()))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	clean_text(clean_text)
	translate(translate)
	summaries(summaries)
	__end__([<p>__end__</p>]):::last
	__start__ --> clean_text;
	clean_text --> translate;
	translate --> summaries;
	summaries --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


In [24]:
graph.invoke({"raw_text": " Hala mundo "})

{'raw_text': ' Hala mundo ',
 'clean': 'Hala mundo',
 'translated': 'The phrase "Hala mundo" can be translated to "Hello world" in English. It is often used in programming and technology contexts, particularly when introducing a new programming language or environment.',
 'summary': 'The phrase "Hala mundo" translates to "Hello world" in English and is commonly used in programming and technology, especially when introducing a new programming language or environment.'}

# Conditional Edge

A router function inspects the state and returns the name of the next node. This is LangGraph's "if-else" — the LLM can decide which branch to take based on what it sees.

In [25]:
class TicketState(TypedDict):
    query: str
    ticket_type: str   # Set by classify
    response: str

In [27]:
def classify(s):
    result = llm.invoke(
        "Classify as 'refund' or 'faq': "
        + s["query"])
    return {"ticket_type": result.content.strip()}

# Router — inspects state, returns node name

In [28]:
def route(s) -> str:
    if s["ticket_type"] == "refund":
        return "handle_refund"
    return "answer_faq"

In [29]:
def handle_refund(s):
    return {"response": "Starting refund..."}

In [30]:
def answer_faq(s):
    return {"response": "Here's the FAQ..."}

In [31]:
builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("handle_refund", handle_refund)
builder.add_node("answer_faq", answer_faq)

In [33]:
builder.add_edge(START, "classify")
# Conditional edge - calls route() to decide
builder.add_conditional_edges(
    "classify", route,
    {"handle_refund": "handle_refund",
    "answer_faq": "answer_faq"})
builder.add_edge("handle_refund", END)
builder.add_edge("answer_faq", END)

In [34]:
graph = builder.compile()

result = graph.invoke({
    "query": "I want a refund for my order"
})

print(result)

{'query': 'I want a refund for my order', 'ticket_type': 'refund', 'response': 'Starting refund...'}


# Loop/Cycle

Cycles are what make LangGraph an "agentic" framework. A node can loop back to an earlier node — perfect for retry logic, reflection, or iterative refinement. A counter guards against infinite loops.

In [36]:
class CodeState(TypedDict):
    task: str
    code: str
    passed: bool
    tries: int  # guards the loop

In [37]:
def generate_code(s):
    code = llm.invoke(s["task"]).content
    return {"code": code,
            "tries": s["tries"] + 1}

In [38]:
def check_quality(s):
    ok = "def " in s["code"]
    return {"passed": ok}

# Router: loop or finish

In [39]:
def should_retry(s) -> str:
    if s["passed"] or s["tries"] >= 3:
        return END
    return "generate_code" # loop back

In [40]:
builder = StateGraph(CodeState)
builder.add_node("generate_code", generate_code)
builder.add_node("check_quality", check_quality)


# Edge
builder.add_edge(START, "generate_code")
builder.add_edge("generate_code", "check_quality")
builder.add_conditional_edges(
    "check_quality", should_retry)

In [41]:
graph = builder.compile()

In [43]:
graph.invoke({"task": "Write fizzbuzz",
            "tries": 0, "passed": False})

{'task': 'Write fizzbuzz',
 'code': 'Certainly! The FizzBuzz problem is a common coding challenge that involves printing numbers from 1 to a specific limit, but with a slight twist:\n\n- For numbers that are multiples of 3, print "Fizz" instead of the number.\n- For numbers that are multiples of 5, print "Buzz" instead of the number.\n- For numbers that are multiples of both 3 and 5, print "FizzBuzz" instead of the number.\n- For all other numbers, just print the number itself.\n\nHere’s a simple implementation in Python:\n\n```python\ndef fizzbuzz(n):\n    for i in range(1, n + 1):\n        if i % 3 == 0 and i % 5 == 0:\n            print("FizzBuzz")\n        elif i % 3 == 0:\n            print("Fizz")\n        elif i % 5 == 0:\n            print("Buzz")\n        else:\n            print(i)\n\n# Call the function with a limit, for example, 100\nfizzbuzz(100)\n```\n\nYou can replace `100` with any upper limit you want to test. This code will print the required output for each number fr